# Compcor Comparison
- Metrics and setup taken from [the compcor library](https://github.com/IBM/comparing-corpora) and the [meme setup](https://github.com/IBM/meme) from ["Measuring the Measuring Tools" by Kour et al.](https://doi.org/10.18653/v1/2022.gem-1.35)

In [31]:
import time
import random
import torch
import os
import glob
import sklearn

import pandas as pd
import polars as pl
import numpy as np


import scipy
import statsmodels.api as sm
from statsmodels.formula.api import ols


import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder
from KSC import KSC

from itertools import combinations, product

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

In [ ]:
def make_input_data(file_paths="./outputsTrain/*/texts_and_ids.csv"):
    # Get all CSV files.
    all_files = glob.glob(file_paths)
    # Loop through files.
    all_dfs = {}
    for f in all_files:
        df = pd.read_csv(f)
        df['category'] = df['doc_id'].apply(lambda x: x.split('_')[0])
        all_dfs[f.split('/')[-2]] = df
    return all_dfs

train_dfs = make_input_data()

In [ ]:
setA = ['can you tell me how i would normally say thank you as a french person', 'can you translate hi into spanish for me', 'can you translate milk into spanish for me', 'how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do germans say goodnight','how do i ask about the weather in chinese', 'how do i say hotel in finnish', 'how do i say bathroom in italian']
setB = ['how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do they say tacos in mexico', 'how do they say yes in brazil', 'how do vietnameses people say hello', 'how do you say cat in spanish', 'how do you say dog in spanish', 'how do you say fast in spanish', 'how do you say good bye in french', 'how do you say goodbye in spanish', 'how do you say hello in french', 'how do you say hello in japanese', 'how do you say hello in mexico']

In [ ]:
def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    distances

In [35]:
for (name1, df1), (name2, df2) in combinations(train_dfs.items(), 2):
    print(name1, name2)
    for categoryA, categoryB in product(df1['category'].value_counts().index.values.to_numpy(), df2['category'].value_counts().index.values.to_numpy()):
        print(f"Label:{categoryA}-{categoryB}")
        # print(category, df1[df1['category'] == category]['text'].values.to_numpy())
        # break
    # print(df1['text'].values.to_numpy(), df2['text'].values.to_numpy())
    break

1618 1619
Label:atis-atis
Label:atis-banking77
Label:atis-clinc150
Label:atis-clinicalDialogueSummarizations
Label:atis-dementiaAudio
Label:atis-huffPostNews
Label:atis-medicalAbstracts
Label:atis-simSUM
Label:atis-syntheticCareHomeNurseNotes
Label:atis-yahoo
Label:banking77-atis
Label:banking77-banking77
Label:banking77-clinc150
Label:banking77-clinicalDialogueSummarizations
Label:banking77-dementiaAudio
Label:banking77-huffPostNews
Label:banking77-medicalAbstracts
Label:banking77-simSUM
Label:banking77-syntheticCareHomeNurseNotes
Label:banking77-yahoo
Label:clinc150-atis
Label:clinc150-banking77
Label:clinc150-clinc150
Label:clinc150-clinicalDialogueSummarizations
Label:clinc150-dementiaAudio
Label:clinc150-huffPostNews
Label:clinc150-medicalAbstracts
Label:clinc150-simSUM
Label:clinc150-syntheticCareHomeNurseNotes
Label:clinc150-yahoo
Label:clinicalDialogueSummarizations-atis
Label:clinicalDialogueSummarizations-banking77
Label:clinicalDialogueSummarizations-clinc150
Label:clinicalD